In [ ]:
Tavily_api_key = 0 # use your own key here
Chat_groq_API = 0 # use your own key here

In [2]:
pip install langchain langgraph langchain-community tavily-python langchain-groq

  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.13.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached mypy_extensions-1.1.0-py3-none-any.whl.metadata (1.1 kB)
Using cached langchain_community-0.4.1-py3-none-any.whl (2.5 MB)
Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
Using cached httpx_sse-0.4.3-py3-none-any.whl (9.0 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ------------------------------ --------- 0.8/1.0 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 4.5 MB/s eta 0:00:00
Using cached marshmallow-3.2

  You can safely remove it manually.


In [ ]:
import os
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults

# ✅ Tavily API Key → Used for real-time web search
os.environ["TAVILY_API_KEY"] = Tavily_api_key

# ✅ Groq API Key → Used for LLM (llama-3.1-8b-instant) to generate answers
os.environ["GROQ_API_KEY"] = Chat_groq_API

class State(TypedDict):
    question: str
    search_results: str
    answer: str

# 🤖 Groq LLM - generates the final answer
llm = ChatGroq(model="llama-3.1-8b-instant")

# 🔍 Tavily Search - fetches real-time web results
tavily = TavilySearchResults(max_results=3)

def search_node(state: State) -> State:
    print("🔍 [Tavily API] Searching the web...")
    results = tavily.invoke(state["question"])
    formatted = "\n".join([r["content"] for r in results])
    return {"question": state["question"], "search_results": formatted, "answer": ""}

def answer_node(state: State) -> State:
    print("🤖 [Groq API] Generating answer...")
    prompt = f"""
    Use the following search results to answer the question:

    Question: {state['question']}

    Search Results:
    {state['search_results']}

    Provide a concise summary.
    """
    response = llm.invoke(prompt)
    return {"question": state["question"], "search_results": state["search_results"], "answer": response.content}

graph = StateGraph(State)
graph.add_node("search", search_node)
graph.add_node("answer", answer_node)
graph.add_edge(START, "search")
graph.add_edge("search", "answer")
graph.add_edge("answer", END)

app = graph.compile()

result = app.invoke({
    "question": "What is the recent status at the Strait of Hormuz?",
    "search_results": "",
    "answer": ""
})

print("\n📰 Answer:\n", result["answer"])

Current Status at the Strait of Hormuz:

- A US naval blockade is in effect targeting maritime traffic to and from Iranian ports.
- The blockade has significantly reduced ship traffic, with 279 ships passing through the strait between February 28 and April 12, a 95% drop from the pre-war average.
- Since a ceasefire took effect on April 8, only 45 ships have entered or exited the strait.
- The US has instructed ships to take an alternative route through the strait north of Larak Island and exit south of it until further notice.
- Some ships have managed to pass through the strait, including three tankers that entered the Gulf on Tuesday.
- Ship-tracking data shows 22 ships have been attacked since the war on Iran began.
